# Perceptron

### Disciplina de Sistemas Inteligentes

### Departamento de Informática e Estatística

### Profa. Nathalia da Cruz Alves


**Nota 1**: Em caso de reprodução, publicação ou compartilhamento deste material, a autoria deverá ser obrigatoriamente mantida.

**Nota 2**: Este material foi elaborado com o auxílio de ferramentas de Inteligência Artificial (IA), utilizadas para o refinamento de explicações e revisão de trechos de código. A autoria e a responsabilidade pelo conteúdo final permanecem com a autora.

Estudante(s): insira seu nome aqui


**Observação**: Solicita-se que sejam informados apenas os nomes dos estudantes efetivamente participantes da atividade.

Nesta aula, você vai explorar o funcionamento das redes neurais na prática, treinando modelos, acompanhando seu processo de aprendizado e visualizando como eles aprendem a separar diferentes classes de dados.

Ao longo da aula, você irá:

- Treinar redes neurais e acompanhar sua evolução durante o aprendizado.

- Visualizar fronteiras de decisão e compreender como os modelos classificam os dados.

- Investigar a separação linear e suas limitações.

- Resolver o problema XOR e entender por que ele exige uma abordagem mais complexa.

- Explorar o papel das camadas ocultas e das funções de ativação na capacidade de aprendizado das redes neurais.




### Configurações iniciais

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# Set da seed das bibliotecas para reprodução de resultados

tf.random.set_seed(42)
np.random.seed(42)

## Primeiro Perceptron

Nesta etapa, você vai treinar um perceptron, o modelo mais simples de uma rede neural, para aprender a operação lógica AND.

O objetivo é fazer com que o neurônio aprenda, a partir de exemplos, a produzir a saída correta para cada combinação de entradas.


Tabela-verdade da função AND:

| A | B | A AND B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    0    |
| 1 | 0 |    0    |
| 1 | 1 |    1    |

Como funciona o aprendizado?
- Entradas: o perceptron recebe os valores A e B.
- Pesos e viés: o modelo utiliza parâmetros ajustáveis para combinar as entradas.
- Função de ativação: determina a saída do neurônio com base no resultado calculado.
- Treinamento: os pesos e o viés são atualizados para reduzir os erros nas previsões.



In [ ]:
# @title Entradas (X) e saída (y)

X = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
])

y = np.array([0,0,0,1])

**Criando o Perceptron**

Vamos construir um perceptron com um único neurônio, capaz de receber duas entradas e produzir uma saída por meio da função de ativação sigmoidal.


**Estrutura do modelo**
- 2 variáveis de entrada: A e B, correspondentes aos valores da função lógica AND.
- 1 neurônio: combina as entradas utilizando pesos e um viés (bias).


**Função de ativação**
- Sigmoidal: transforma o resultado em um valor entre 0 e 1. É suave, contínua e derivável, permitindo o treinamento da rede neural com gradiente (derivadas) descendente.


**Função de loss (perda)**
- É uma função matemática que calcula e penaliza o erro de uma forma específica para ajudar a rede neural a aprender.
- A binary_crossentropy compara a probabilidade prevista pelo modelo com a resposta real (0 ou 1). O objetivo dela é penalizar o modelo quando ele tem muita certeza de uma resposta errada.

In [ ]:
# @title Função para criar e treinar um Perceptron simples

def treinar_perceptron(
    X,                      # dados de entrada
    y,                      # saídas desejadas
    ativacao='sigmoid',     # função de ativação
    learning_rate=0.1,      # taxa de aprendizagem
    epochs=1000             # número de épocas
):
    """
    Cria e treina um Perceptron simples usando TensorFlow.

    Retorna:
    - model   -> rede neural treinada
    - history -> histórico do treinamento
    """

    # Cria um modelo sequencial, i.e., camadas organizadas em sequência
    model = tf.keras.Sequential([
        # Adiciona uma camada densa (Dense)
        tf.keras.layers.Dense(
            1,                    # número de neurônios da camada
            activation=ativacao,  # função de ativação
            input_shape=(2,)      # quantidade de entradas (x1 e x2)
        )
    ])

    # Configura o processo de treinamento
    model.compile(
        # Algoritmo de otimização
        optimizer=tf.keras.optimizers.SGD(
            # Taxa de aprendizagem
            learning_rate=learning_rate
        ),
        # Função de loss: penalização do tamanho do erro durante o treinamento
        loss='binary_crossentropy',
        # Métrica usada para avaliação para classificação (0 ou 1)
        metrics=['accuracy']
    )

    # Treina a rede neural
    history = model.fit(
        X,              # entradas
        y,              # saídas desejadas
        epochs=epochs,  # quantidade de épocas
        verbose=0       # não mostrar logs durante o treinamento
    )

    # Retorna o modelo treinado e o histórico
    return model, history


model, history = treinar_perceptron(X, y)

### Mostrar previsões

- Após definir a estrutura do neurônio e treiná-lo, vamos testar suas previsões para todas as combinações de entradas da função lógica AND.

- Antes do treinamento, as previsões dependiam dos pesos e do viés inicializados. Após o treinamento, esperamos que o modelo produza valores próximos das saídas desejadas.

- Como estamos utilizando uma função de ativação sigmoidal, as previsões serão valores entre 0 e 1. Esperamos que, após o treinamento, as saídas se aproximem de 0 para as combinações falsas e de 1 para a combinação [1, 1].




In [ ]:
preds = model.predict(X)

for entrada, pred in zip(X, preds):
    classe = int(pred[0] >= 0.5)
    print(
        f"Entrada: {entrada} -> "
        f"previsão: {pred[0]:.4f} | "
        f"classe prevista: {classe}"
    )

Pergunta: por que as previsões da rede são valores contínuos
próximos de 0 e 1, e não exatamente binários?

### Loss (perda) e Acurácia

Quando treinamos uma rede neural, coletamos as métricas de desempenho a cada época (uma volta completa de treinamento com todas as amostras de dados).

É importante visualizar o comportamento de algumas métricas do modelo ao longo do treinamento.

- Gráfico de loss: Mostra o valor da função de perda (nesse caso, binary_crossentropy). O objetivo do modelo é fazer essa curva cair continuamente, indicando que ele está errando cada vez menos.

- Gráfico de acurácia: Mostra a porcentagem de acertos do modelo (de 0 a 100%, ou de 0 a 1). O objetivo é ver a curva subir e se aproximar do topo (1.0), indicando que o modelo está classificando os dados corretamente.

In [ ]:
# @title Visualizando loss e acurácia durante o treinamento

def plot_metrics(history):
    """
    Exibe os gráficos de loss e acurácia
    ao longo das épocas de treinamento.
    """

    # Cria dois gráficos lado a lado
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))

    # Gráfico de loss
    ax[0].plot(
        history.history['loss'],
        color='blue',
        linewidth=2
    )

    ax[0].set_title("Loss durante o treinamento")
    ax[0].set_xlabel("Época")
    ax[0].set_ylabel("Loss")
    ax[0].grid(True)

    # Gráfico de acurácia
    ax[1].plot(
        history.history['accuracy'],
        color='green',
        linewidth=2
    )

    ax[1].set_title("Acurácia durante o treinamento")
    ax[1].set_xlabel("Época")
    ax[1].set_ylabel("Acurácia")
    ax[1].set_ylim(0, 1.05)
    ax[1].grid(True)

    # Ajusta o layout e exibe os gráficos
    plt.tight_layout()
    plt.show()


plot_metrics(history)

Perguntas:

- A perda aumenta ou diminui?

- O que acontece nas primeiras épocas?

- O modelo “aprendeu”?

### Visualizando a fronteira de decisão

A fronteira de decisão do modelo pode ser observada por meio de um "mapa colorido" que mostra exatamente como a rede neural está dividindo o espaço para tentar separar as classes 0 e 1.

In [ ]:
# @title Função para observar a fronteira de decisão

def plot_boundary(model):
    xx, yy = np.meshgrid(
        np.linspace(-0.5, 1.5, 200),
        np.linspace(-0.5, 1.5, 200)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    preds = model.predict(grid)
    preds = preds.reshape(xx.shape)

    plt.contourf(xx, yy, preds, levels=20, cmap="coolwarm", alpha=0.6)
    plt.scatter(X[:,0], X[:,1], c=y, s=120, edgecolors='k')

    plt.title("Fronteira de decisão")
    plt.show()

plot_boundary(model)

## Exercício 1 - Perceptron para a função OU

Nesta etapa, você deve treinar um perceptron para aprender a operação lógica OU.

Tabela-verdade da função OR:

| A | B | A OR B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    1    |
| 1 | 0 |    1    |
| 1 | 1 |    1    |


Ao final, mostre:
- As predições do modelo
- Os gráficos de loss e acurácia
- A visualização da fronteira de decisão

## Exercício 2 - Perceptron para a função XOR

Até agora, o Perceptron conseguiu aprender corretamente funções como AND e OR.
Mas será que um único neurônio artificial consegue resolver qualquer problema?

Nesta etapa, você deve testar o treinamento da função lógica XOR (OU-EXCLUSIVO), um problema clássico da história das redes neurais.

Tabela-verdade da função XOR:

| A | B | A XOR B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    1    |
| 1 | 0 |    1    |
| 1 | 1 |    0    |

Ao final, mostre:
- As predições do modelo
- Os gráficos de loss e acurácia
- A visualização da fronteira de decisão

Perguntas:

- O modelo conseguiu aprender corretamente a função XOR?
- A taxa de erro ficou próxima de zero?
- A fronteira de decisão criada pelo Perceptron consegue separar corretamente as duas classes?
- Por que uma única reta não é suficiente para resolver esse problema?
- O que isso nos diz sobre as limitações do Perceptron simples?

Tente modificar:

- a quantidade de épocas
- a taxa de aprendizagem
- a função de ativação

O modelo consegue resolver o XOR apenas com um único neurônio? Por quê?

## Multilayer Perceptron (MLP)

No exercício anterior, vimos que um único Perceptron não consegue resolver o problema XOR.

Isso acontece porque o XOR não é linearmente separável:
não existe uma única reta capaz de separar corretamente as classes.

Para superar essa limitação, vamos utilizar uma rede neural com:

- uma camada oculta
- múltiplos neurônios
- função de ativação não linear tanh

**Função de ativação não linear tanh**

- A função tanh (tangente hiperbólica) mapeia as saídas entre -1 e 1, sendo simétrica em relação à origem.

- Para problemas lógicos pequenos como o XOR, ela costuma se comportar de forma muito mais estável que outras funções, como ReLU (que pode "desligar" neurônios se os pesos ficarem negativos no começo) e sigmoid (que "esmaga" qualquer entrada para um valor entre 0 e 1, podendo fazer com que a derivada da sigmoid fique muito próxima de zero levando ao problema do gradiente evanescente/vanishing gradients).


As camadas ocultas permitem que diferentes neurônios se especializem em diferentes regiões do espaço de entrada, possibilitando a criação de fronteiras de decisão mais complexas.

Essa arquitetura é chamada de MLP (MultiLayer Perceptron).


### XOR com MLP
Treine um novo modelo para a função OU-EXCLUSIVO

| A | B | A XOR B |
|---|---|---------|
| 0 | 0 |    0    |
| 0 | 1 |    1    |
| 1 | 0 |    1    |
| 1 | 1 |    0    |



Durante a execução, observe:

- o comportamento do erro
- as previsões da rede
- a nova fronteira de decisão aprendida


In [ ]:
# @title Função para criar uma MLP

def treinar_mlp(
    X,
    y,
    neuronios_ocultos,
    ativacao='tanh',
    learning_rate=0.1,
    epochs=1000):

    tf.keras.backend.clear_session()

    # Cria o modelo
    model = tf.keras.Sequential([

        # Camada oculta
        tf.keras.layers.Dense(
            neuronios_ocultos,
            activation=ativacao,
            input_shape=(2,)
        ),

        # Camada de saída
        tf.keras.layers.Dense(
            1,
            activation='sigmoid'
        )
    ])

    # Compila a rede
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Treina a rede
    history = model.fit(
        X,
        y,
        epochs=epochs,
        verbose=0
    )

    return model, history

**Observação**: a inicialização dos pesos do Keras é aleatória por padrão, muitas vezes a rede pode cair em mínimos locais ruins onde ela não consegue separar o XOR corretamente com apenas 2 neurônios ocultos, resultando em fronteiras de decisão incorretas. Se mesmo com 1000 épocas a fronteira ainda falhar, rode a célula de treino novamente, que criará uma nova inicialização aleatória.

In [ ]:
# Dados XOR
X = np.array([
    [0,0],
    [0,1],
    [1,0],
    [1,1]
])

y = np.array([0,1,1,0])

# Treina a MLP
model, history = treinar_mlp(X, y, neuronios_ocultos=2)

# Mostra previsões
preds = model.predict(X)

print("Predições da MLP:")

for entrada, pred in zip(X, preds):
    classe = int(pred[0] >= 0.5)
    print(
        f"Entrada: {entrada} -> "
        f"previsão: {pred[0]:.4f} | "
        f"classe prevista: {classe}"
    )

plot_metrics(history)

plot_boundary(model)

Após executar o treinamento da MLP, responda:

- A rede conseguiu aprender corretamente o XOR?
- O erro final foi menor do que no Perceptron simples?
- O que mudou na fronteira de decisão?
- Por que adicionar neurônios intermediários aumenta a capacidade da rede?
- Qual o papel da função de ativação não linear nesse processo?

Nesta aula prática, vimos que:

- um Perceptron simples aprende problemas linearmente separáveis
- XOR não pode ser resolvido por uma única reta
- camadas ocultas aumentam a capacidade da rede
- funções de ativação não lineares são essenciais
- MLPs conseguem aprender padrões mais complexos

Esses conceitos são a base das redes neurais modernas.
